# Scattering rates
Acoustic-phonon scattering rates of electrons and holes: maximum over the band pairs of the
emission and absorption rates (interpolated on a common energy grid), compared with the
HSE benchmark of `data_comparisons/PBE-aco-scattering.mat`.
Figures are saved in `save_path` as `.png` and editable `.mplfig`.

In [ ]:
import os

import matplotlib.pyplot as plt
import numpy as np
import scipy.io as spio
from scipy.interpolate import interp1d

from mcplot import save_figure

data_path = os.path.expanduser("~/mnt/server/material_lib/silicon/scat_rates_txt")
save_path = "fig/"

type_scattering = "aco"
name = "TA"
bands_e = [5, 6, 7, 8, 9, 10, 11]  # initial and final bands
bands_h = [1, 2, 3, 4]

# Interpolation grid (eV)
delta_eV = 1e-3
x_max = 15
x_interp = np.linspace(0, x_max, int(x_max / delta_eV))

# Load data

In [ ]:
def smooth(y, span):
    """Moving average over `span` points (like MATLAB smooth(y, span))."""
    return np.convolve(y, np.ones(span) / span, mode="same")


def max_rate(specie, process, bands):
    """Maximum over all (initial, final) band pairs of the rate interpolated on x_interp.

    process: "emi" or "abs". Missing files and NaNs count as zero rate.
    """
    print(f">> Reading {specie} {process}!")
    rates = np.zeros((len(bands) ** 2, len(x_interp)))
    for row, (b1, b2) in enumerate((b1, b2) for b1 in bands for b2 in bands):
        filename = f"{data_path}/{specie}_{type_scattering}{process}_{name}_band_{b1}_band_{b2}.txt"
        print(f"Reading: {filename}...")
        if not os.path.exists(filename):
            print("Skip!")
            continue
        try:
            data = np.loadtxt(filename)
        except Exception as e:
            print(f"Error reading {filename}: {e}")
            continue
        if data.ndim == 2 and data.shape[1] >= 2:
            rates[row] = interp1d(data[:, 0], data[:, 1], bounds_error=False, fill_value=np.nan)(x_interp)
        else:
            print(f"Invalid data in {filename}!")
    return np.nan_to_num(rates, nan=0.0).max(axis=0)


# Benchmark (HSE)
try:
    mat_data = spio.loadmat("data_comparisons/PBE-aco-scattering.mat")
    elec_aco, hole_aco = mat_data["elec_aco"], mat_data["hole_aco"]
except FileNotFoundError:
    print("Warning: 'PBE-aco-scattering.mat' not found, the benchmark curve is not plotted.")
    elec_aco = hole_aco = None

rates = {}
for specie, bands in (("e", bands_e), ("h", bands_h)):
    emi = max_rate(specie, "emi", bands)
    abs_ = max_rate(specie, "abs", bands)
    rates[specie] = {"emi": emi, "abs": abs_, "total": emi + abs_}
    print(f"{specie}: done.")

# Plots

In [ ]:
plt.rcParams.update({"font.size": 14, "axes.linewidth": 1})


def rate_figure(specie, benchmark, title, xlim, ylim, energy_unit="eV", log_x=False):
    """Emission, absorption and total rate with the smoothed benchmark (if available)."""
    scale = 1e3 if energy_unit == "meV" else 1.0
    r = rates[specie]
    fig, ax = plt.subplots(figsize=(7, 6))
    ax.plot(x_interp * scale, r["emi"], linewidth=1.5, color="blue", label="Emission Total Rate")
    ax.plot(x_interp * scale, r["abs"], linewidth=1.5, color="blue", label="Absorption Total Rate")
    ax.plot(x_interp * scale, r["total"], linewidth=2, color="red", label="Total Rate")
    if benchmark is not None:
        ax.plot(benchmark[:, 0] * scale, smooth(benchmark[:, 1] * 1e12, 99), linewidth=1.5, color="black",
                label="HSE Rate")
    ax.grid(True)
    ax.set_xlabel(f"Energy, {energy_unit}")
    ax.set_ylabel("Rate, 1/s")
    ax.set_title(title)
    ax.set_yscale("log")
    if log_x:
        ax.set_xscale("log")
    ax.set_xlim(*xlim)
    ax.set_ylim(*ylim)
    ax.legend(loc="upper right", fontsize=11)
    fig.tight_layout()
    return fig


figures = {
    "e_aco_scattering_rate": rate_figure("e", elec_aco, "Electrons", (0, x_max), (1e12, 2e15)),
    "e_aco_scattering_rate_detail": rate_figure("e", elec_aco, "Electrons", (0.1, 1000), (1e10, 2e13),
                                                energy_unit="meV", log_x=True),
    "h_aco_scattering_rate": rate_figure("h", hole_aco, "Holes", (0, x_max), (1e12, 2e15)),
    "h_aco_scattering_rate_detail": rate_figure("h", hole_aco, "Holes", (0, 0.5), (1e11, 1e14)),
}
for fname, fig in figures.items():
    save_figure(fig, os.path.join(save_path, fname), formats=("png",))
plt.show()